### Cell 1 — Setup

In [1]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

DATA_DIR = Path('../data/processed')
RESULTS_DIR = Path('../results')
TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
MODELS_DIR = RESULTS_DIR / 'models'

RANDOM_STATE = 42

# Load the encoded permissive data (we need numeric encoding for MLP)
df_encoded = pd.read_csv(DATA_DIR / 'metabric_permissive_encoded.csv')

# Load split indices
splits = pd.read_csv(DATA_DIR / 'split_indices.csv')
train_idx = splits['permissive_train'].dropna().astype(int).values
test_idx = splits['permissive_test'].dropna().astype(int).values

X = df_encoded.drop(columns=['Relapse Free Status']).astype(float)
y = df_encoded['Relapse Free Status'].astype(int)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Load the canonical GBM (saved from Phase 2)
with open(MODELS_DIR / 'canonical_gbm.pkl', 'rb') as f:
    gbm_data = pickle.load(f)
gbm = gbm_data['model']

# Sanity check: the GBM should reproduce its Phase 2 numbers
gbm_pred = gbm.predict(X_test)
print(f"Canonical GBM (Phase 2 reload):")
print(f"  Test accuracy: {accuracy_score(y_test, gbm_pred):.3f}")
print(f"  Test recall:   {recall_score(y_test, gbm_pred, pos_label=1):.3f}")

Canonical GBM (Phase 2 reload):
  Test accuracy: 0.665
  Test recall:   0.344


### Cell 2 — Train an MLP

In [2]:
# === Train an MLP for the second mimic-tree experiment ===
# Small architecture (one hidden layer of 32) — enough capacity for this size
# of dataset without being needlessly complex.

mlp = MLPClassifier(
    hidden_layer_sizes=(32,),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=RANDOM_STATE,
    early_stopping=True,
    validation_fraction=0.1,
)
mlp.fit(X_train, y_train)

mlp_pred = mlp.predict(X_test)
print(f"MLP baseline:")
print(f"  Test accuracy: {accuracy_score(y_test, mlp_pred):.3f}")
print(f"  Test recall:   {recall_score(y_test, mlp_pred, pos_label=1):.3f}")
print(f"  Converged in {mlp.n_iter_} iterations")

# Save the MLP for later use (Phase 6 may want it as a starting point)
with open(MODELS_DIR / 'baseline_mlp.pkl', 'wb') as f:
    pickle.dump({
        'model': mlp,
        'features': X_train.columns.tolist(),
    }, f)

MLP baseline:
  Test accuracy: 0.657
  Test recall:   0.338
  Converged in 19 iterations


### Cell 3 — Helper for mimic tree evaluation

In [3]:
def evaluate_mimic_tree(parent_model, parent_name, max_depth=4):
    """
    Train a decision tree on the parent's hard predictions on training data.
    Evaluate fidelity (agreement with parent on test set) and true accuracy
    (agreement with actual labels on test set).
    """
    # Get parent's predictions on training set
    parent_train_preds = parent_model.predict(X_train)
    
    # Train the mimic tree
    mimic = DecisionTreeClassifier(
        max_depth=max_depth,
        random_state=RANDOM_STATE,
    )
    mimic.fit(X_train, parent_train_preds)
    
    # Get parent's predictions on test set (for fidelity)
    parent_test_preds = parent_model.predict(X_test)
    mimic_test_preds = mimic.predict(X_test)
    
    # Fidelity: do mimic and parent agree?
    fidelity = (mimic_test_preds == parent_test_preds).mean()
    
    # True accuracy: how does mimic do against actual labels?
    true_acc = accuracy_score(y_test, mimic_test_preds)
    true_recall = recall_score(y_test, mimic_test_preds, pos_label=1)
    
    # Tree complexity
    n_leaves = mimic.get_n_leaves()
    n_features_used = (mimic.feature_importances_ > 0).sum()
    
    tn, fp, fn, tp = confusion_matrix(y_test, mimic_test_preds).ravel()
    
    return {
        'parent_model': parent_name,
        'mimic_max_depth': max_depth,
        'fidelity_to_parent': fidelity,
        'true_test_accuracy': true_acc,
        'true_test_recall': true_recall,
        'n_leaves': n_leaves,
        'n_features_used': int(n_features_used),
        'parent_test_accuracy': accuracy_score(y_test, parent_test_preds),
        'parent_test_recall': recall_score(y_test, parent_test_preds, pos_label=1),
        'test_true_neg': tn, 'test_false_pos': fp,
        'test_false_neg': fn, 'test_true_pos': tp,
    }, mimic


def print_mimic_metrics(m):
    print(f"\n{'='*60}")
    print(f"  Mimic tree of {m['parent_model']} (max_depth={m['mimic_max_depth']})")
    print('='*60)
    print(f"  Parent's test accuracy: {m['parent_test_accuracy']:.3f}")
    print(f"  Parent's test recall:   {m['parent_test_recall']:.3f}")
    print(f"  Mimic's true test accuracy: {m['true_test_accuracy']:.3f}")
    print(f"  Mimic's true test recall:   {m['true_test_recall']:.3f}")
    print(f"  Fidelity to parent:     {m['fidelity_to_parent']:.3f}")
    print(f"  Tree size: {m['n_leaves']} leaves, {m['n_features_used']} features used")
    print(f"  Confusion: TN={m['test_true_neg']}, FP={m['test_false_pos']}, "
          f"FN={m['test_false_neg']}, TP={m['test_true_pos']}")


all_results = []

### Cell 4 — Mimic the GBM

In [4]:
# === Mimic tree from the canonical GBM ===

m_gbm, mimic_gbm = evaluate_mimic_tree(gbm, "Canonical GBM (full features)", max_depth=4)
print_mimic_metrics(m_gbm)
all_results.append(m_gbm)

# Print the tree as readable rules
print("\nMimic tree from GBM (text form):")
tree_text = export_text(mimic_gbm, feature_names=X_train.columns.tolist(), max_depth=4)
print(tree_text)


  Mimic tree of Canonical GBM (full features) (max_depth=4)
  Parent's test accuracy: 0.665
  Parent's test recall:   0.344
  Mimic's true test accuracy: 0.652
  Mimic's true test recall:   0.362
  Fidelity to parent:     0.942
  Tree size: 15 leaves, 8 features used
  Confusion: TN=199, FP=35, FN=102, TP=58

Mimic tree from GBM (text form):
|--- Lymph nodes examined positive <= -0.11
|   |--- Nottingham prognostic index <= 0.90
|   |   |--- HER2 Status_Negative <= 0.50
|   |   |   |--- Tumor Size(mm) <= 0.07
|   |   |   |   |--- class: 0
|   |   |   |--- Tumor Size(mm) >  0.07
|   |   |   |   |--- class: 0
|   |   |--- HER2 Status_Negative >  0.50
|   |   |   |--- class: 0
|   |--- Nottingham prognostic index >  0.90
|   |   |--- Age at Diagnosis <= 0.52
|   |   |   |--- Nottingham prognostic index <= 0.93
|   |   |   |   |--- class: 1
|   |   |   |--- Nottingham prognostic index >  0.93
|   |   |   |   |--- class: 0
|   |   |--- Age at Diagnosis >  0.52
|   |   |   |--- HER2 Status_

### Cell 5 — Mimic the MLP

In [5]:
# === Mimic tree from the MLP ===

m_mlp, mimic_mlp = evaluate_mimic_tree(mlp, "Baseline MLP", max_depth=4)
print_mimic_metrics(m_mlp)
all_results.append(m_mlp)

# Print the tree as readable rules
print("\nMimic tree from MLP (text form):")
tree_text = export_text(mimic_mlp, feature_names=X_train.columns.tolist(), max_depth=4)
print(tree_text)


  Mimic tree of Baseline MLP (max_depth=4)
  Parent's test accuracy: 0.657
  Parent's test recall:   0.338
  Mimic's true test accuracy: 0.650
  Mimic's true test recall:   0.287
  Fidelity to parent:     0.921
  Tree size: 16 leaves, 8 features used
  Confusion: TN=210, FP=24, FN=114, TP=46

Mimic tree from MLP (text form):
|--- Nottingham prognostic index <= 0.89
|   |--- Nottingham prognostic index <= 0.05
|   |   |--- Age at Diagnosis <= -0.90
|   |   |   |--- HER2 Status_Positive <= 0.50
|   |   |   |   |--- class: 0
|   |   |   |--- HER2 Status_Positive >  0.50
|   |   |   |   |--- class: 1
|   |   |--- Age at Diagnosis >  -0.90
|   |   |   |--- Nottingham prognostic index <= 0.01
|   |   |   |   |--- class: 0
|   |   |   |--- Nottingham prognostic index >  0.01
|   |   |   |   |--- class: 0
|   |--- Nottingham prognostic index >  0.05
|   |   |--- Lymph nodes examined positive <= 0.14
|   |   |   |--- HER2 Status_Positive <= 0.50
|   |   |   |   |--- class: 0
|   |   |   |--- H

### Cell 6 — Compare mimic trees against GOSDT

In [6]:
# === Compare against GOSDT ===
# Load the GOSDT model from Phase 4 to compare directly

with open(MODELS_DIR / 'gosdt_canonical.pkl', 'rb') as f:
    gosdt_data = pickle.load(f)

# GOSDT was trained on binarized data, so we need that for predictions
df_binarized = pd.read_csv(DATA_DIR / 'metabric_permissive_binarized.csv')
X_binarized = df_binarized.drop(columns=['target']).astype(int)
X_binarized_test = X_binarized.iloc[test_idx]
y_binarized = df_binarized['target'].astype(int)
y_binarized_test = y_binarized.iloc[test_idx]

gosdt_pred = gosdt_data['model'].predict(X_binarized_test)
gosdt_acc = accuracy_score(y_binarized_test, gosdt_pred)
gosdt_recall = recall_score(y_binarized_test, gosdt_pred, pos_label=1)

print("="*60)
print("  COMPARISON: GOSDT vs mimic trees")
print("="*60)
print()
print(f"GOSDT (Phase 4):")
print(f"  Test accuracy: {gosdt_acc:.3f}")
print(f"  Test recall:   {gosdt_recall:.3f}")
print(f"  Tree size: 4 leaves, 3 features used")
print()
print(f"Mimic of GBM (max_depth=4):")
print(f"  Test accuracy: {m_gbm['true_test_accuracy']:.3f}")
print(f"  Test recall:   {m_gbm['true_test_recall']:.3f}")
print(f"  Tree size: {m_gbm['n_leaves']} leaves, {m_gbm['n_features_used']} features used")
print(f"  Fidelity to parent: {m_gbm['fidelity_to_parent']:.3f}")
print()
print(f"Mimic of MLP (max_depth=4):")
print(f"  Test accuracy: {m_mlp['true_test_accuracy']:.3f}")
print(f"  Test recall:   {m_mlp['true_test_recall']:.3f}")
print(f"  Tree size: {m_mlp['n_leaves']} leaves, {m_mlp['n_features_used']} features used")
print(f"  Fidelity to parent: {m_mlp['fidelity_to_parent']:.3f}")

  COMPARISON: GOSDT vs mimic trees

GOSDT (Phase 4):
  Test accuracy: 0.650
  Test recall:   0.375
  Tree size: 4 leaves, 3 features used

Mimic of GBM (max_depth=4):
  Test accuracy: 0.652
  Test recall:   0.362
  Tree size: 15 leaves, 8 features used
  Fidelity to parent: 0.942

Mimic of MLP (max_depth=4):
  Test accuracy: 0.650
  Test recall:   0.287
  Tree size: 16 leaves, 8 features used
  Fidelity to parent: 0.921


c:\Users\liamt\Documents\GitHub\NSAI\final_memo\.nsai-final-env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


### Cell 7 — Save results

In [7]:
# === Save the comparison table ===
results_df = pd.DataFrame(all_results)

numeric_cols = results_df.select_dtypes(include='float').columns
results_df[numeric_cols] = results_df[numeric_cols].round(3)

results_df.to_csv(TABLES_DIR / 'mimic_trees.csv', index=False)
print(f"Saved mimic tree results to results/tables/mimic_trees.csv")
print()
print(results_df.to_string(index=False))

Saved mimic tree results to results/tables/mimic_trees.csv

                 parent_model  mimic_max_depth  fidelity_to_parent  true_test_accuracy  true_test_recall  n_leaves  n_features_used  parent_test_accuracy  parent_test_recall  test_true_neg  test_false_pos  test_false_neg  test_true_pos
Canonical GBM (full features)                4               0.942               0.652             0.362        15                8                 0.665               0.344            199              35             102             58
                 Baseline MLP                4               0.921               0.650             0.288        16                8                 0.657               0.338            210              24             114             46


## Phase 5 Summary: Mimic trees

### What was tested

Two post-hoc interpretability extractions:
1. **DecisionTreeClassifier(max_depth=4) trained on GBM's predictions** — extracts an interpretable tree that mimics the canonical Phase 2 GBM
2. **DecisionTreeClassifier(max_depth=4) trained on MLP's predictions** — same extraction from a single-hidden-layer MLP (32 units, ReLU)

For each mimic tree, we measured:
- **Fidelity to parent**: how often the mimic agrees with the parent on test set
- **True test accuracy**: how the mimic compares as a model in its own right
- **Tree size and feature count**: complexity of the extracted explanation

We compared all of this to GOSDT (Phase 4), which trains an interpretable tree directly on labels rather than extracting one from a parent.

### Headline numbers

| Model | Test Acc | Test Recall | Tree Size | Features Used | Fidelity to Parent |
|---|---|---|---|---|---|
| Canonical GBM (parent) | 0.665 | 0.344 | 200 trees | many | — |
| Baseline MLP (parent) | 0.657 | 0.338 | 32 hidden units | all | — |
| **GOSDT (direct, Phase 4)** | **0.650** | **0.375** | **4 leaves** | **3** | — |
| Mimic of GBM | 0.652 | 0.362 | 15 leaves | 8 | 0.942 |
| Mimic of MLP | 0.650 | 0.287 | 16 leaves | 8 | 0.921 |

### Key findings for the memo

**1. Three interpretable trees, three nearly-identical accuracies.** GOSDT, GBM mimic, and MLP mimic all land within 0.002 accuracy of each other. The dataset's predictive ceiling (~65% accuracy regardless of model class) means that "an interpretable tree" is essentially "an interpretable tree" on this problem — what differs is *how that tree is arrived at*, not what it can achieve.

**2. GOSDT achieves this accuracy with substantially simpler structure.** The mimic trees use 15-16 leaves and 8 features. GOSDT uses 4 leaves and 3 features. **When the goal is an interpretable model, training one directly via GOSDT produces a more compact and accurate tree than extracting one from a black-box parent.**

**3. GOSDT also wins on recall.** 0.375 vs 0.362 (GBM mimic) vs 0.287 (MLP mimic). The MLP mimic in particular is weak on recall — examining its tree, it predicts class 0 (no recurrence) on most paths, giving up substantial recall to maintain precision.

**4. Mimic fidelity is high but imperfect.** GBM mimic agrees with the GBM 94.2% of the time on test; MLP mimic agrees 92.1%. Even at depth 4, the mimic doesn't perfectly capture the parent — it disagrees on roughly 7-8% of borderline cases. This means a "decision tree explanation of a black-box model" is genuinely an *approximation* of the model, not a translation of it.

**5. The mimic trees use different features than GOSDT does.** The GBM mimic uses NPI prominently (it's the second-level split in two of four branches), while GOSDT didn't pick NPI at all. NPI is a composite of tumor size, lymph nodes, and grade — GOSDT chose the underlying features, the mimic chose the composite. Same information, two different ways of slicing it. The MLP mimic shows yet a third pattern, with NPI as the *root* split.

### The "low predictive ceiling" claim is now extremely well-supported

Across four model classes — logistic regression, GBM, MLP, and GOSDT — test accuracy clusters in a 1.5-percentage-point band (0.650-0.665). Sparse, dense, linear, nonlinear, tree-based, neural — they all land essentially at the same accuracy. **This is the strongest possible evidence that ~65% accuracy is the floor of plausibility on this dataset given the available features**, and that the choice between methods on this problem is genuinely about properties other than accuracy.

### Implications for the memo

The Phase 5 finding fits in a single paragraph for the memo:

*"We tested two post-hoc interpretability approaches: extracting decision tree mimics from the canonical GBM and from a baseline MLP. Both mimics achieved high fidelity to their parents (94% and 92% respectively) and true test accuracy comparable to GOSDT's directly-trained tree (0.652 and 0.650 vs GOSDT's 0.650). However, the mimic trees were 4× larger (15-16 leaves vs GOSDT's 4) and used 2-3× as many features (8 vs 3). When the goal is an interpretable model on this dataset, training one directly via GOSDT produces a more compact and accurate tree than extracting one from a black-box parent."*

### Files produced

- `results/tables/mimic_trees.csv`: comparison table
- `results/models/baseline_mlp.pkl`: the MLP (potentially useful for Phase 6 as a starting point)
- The trees printed as text in the notebook for reference

### What's not in this notebook

The Patient 81-style finding from Wisconsin (faithful explanations of wrong predictions) didn't surface here because Phase 5 focused on aggregate metrics. That comparison will surface naturally in Phase 7 (SHAP) where we look at individual misclassifications.